# RSNA Knee: first GPU training run

Run all cells with a Kaggle GPU and Internet enabled. The competition data is attached; no API token belongs in this notebook.

This pilot uses only the 58 studies with complete labels: 43 for training and 15 for validation. It extracts frozen ImageNet ResNet18 features from three slices in each of up to three imaging planes, then fits a 12-label classifier. Missing labels are not converted to negatives. Reports are not used as inference inputs.

Outputs: baseline.pt, metrics.json, history.csv, split.csv, validation_predictions.csv. This is a pipeline baseline, not a competitive model or a submission notebook. Validation on 15 studies is highly uncertain. For a stronger model, use report-derived weak labels for training, retain independently labeled validation, and increase MRI coverage.

Official data: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data
ResNet18: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html
DICOM: https://pydicom.github.io/pydicom/stable/guides/user/image_data_handlers.html


## 1. Environment: run on Kaggle with GPU and Internet enabled.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import subprocess
import sys

missing = [package for module, package in [("pydicom", "pydicom"), ("gdcm", "python-gdcm")]
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import pydicom
import torch
from sklearn.metrics import roc_auc_score
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

SEED = 42
EPOCHS = 100
PLANES = ("Sagittal", "Coronal", "Axial")
torch.manual_seed(SEED)
np.random.seed(SEED)
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Notebook Settings."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

candidates = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")]
DATA = next((path for path in candidates if (path / "train.csv").exists()), None)
assert DATA is not None, "Attach the RSNA Knee Abnormality Detection competition data."
OUT = Path("/kaggle/working")
OUT.mkdir(exist_ok=True)



## 2. Read metadata; missing labels remain unknown, never negative.

In [ ]:
train = pd.read_csv(DATA / "train.csv")
series = pd.read_csv(DATA / "train_series.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
labels = sample.columns.drop("StudyInstanceUID").tolist()
assert len(labels) == 12 and train.StudyInstanceUID.is_unique
assert series.SeriesInstanceUID.is_unique
assert set(series.StudyInstanceUID) <= set(train.StudyInstanceUID)
assert train[labels].stack().isin([0, 1]).all()
gold = train.loc[train[labels].notna().all(axis=1)].copy()
assert len(gold) >= 8, "Too few completely labeled studies for this pilot."
print({"studies": len(train), "series": len(series), "fully_labeled": len(gold),
       "reports": int(train.Report.notna().sum())})
display(pd.DataFrame({"labeled": train[labels].count(), "positive": train[labels].sum()}))

# Split studies before fitting any normalization or classifier.
gold["split_key"] = gold.StudyInstanceUID.map(
    lambda uid: hashlib.sha256(f"{SEED}:{uid}".encode()).hexdigest())
gold = gold.sort_values("split_key").reset_index(drop=True)
n_valid = math.ceil(len(gold) * 0.25)
valid_idx = np.arange(n_valid)
train_idx = np.arange(n_valid, len(gold))
y = gold[labels].to_numpy(dtype=np.float32)
for idx in (train_idx, valid_idx):
    assert ((y[idx].sum(axis=0) > 0) & (y[idx].sum(axis=0) < len(idx))).all(), \
        "Each label needs positive and negative studies in both splits."
assert set(gold.iloc[train_idx].StudyInstanceUID).isdisjoint(gold.iloc[valid_idx].StudyInstanceUID)
split = gold[["StudyInstanceUID"]].copy()
split["split"] = np.where(np.arange(len(gold)) < n_valid, "validation", "train")
split.to_csv(OUT / "split.csv", index=False)
print(f"Train: {len(train_idx)} studies; validation: {len(valid_idx)} studies")



## 3. DICOM decoding and frozen image features.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
transform = weights.transforms()
encoder = resnet18(weights=weights)
encoder.fc = nn.Identity()
encoder = encoder.to(device).eval()
encoder.requires_grad_(False)


def ordered_slices(folder):
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(path, stop_before_pixels=True,
                              specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
               for path in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        assert np.linalg.norm(normal) > 0.9, "Invalid DICOM orientation."
        positions = [float(np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal)) for h in headers]
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        positions = [float(h.InstanceNumber) for h in headers]
    else:
        raise ValueError(f"Missing slice position and InstanceNumber: {folder}")
    return [paths[i] for i in np.argsort(positions, kind="stable")]


def image_tensor(path):
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    assert pixels.ndim == 2 and np.isfinite(pixels).all(), f"Invalid MRI slice: {path}"
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant-valued MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    tensor = torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1)
    return transform(tensor)


@torch.inference_mode()
def study_features(study_uid, series_table, image_root):
    rows = series_table.loc[series_table.StudyInstanceUID == study_uid]
    assert len(rows), f"No series metadata for {study_uid}"
    features = []
    used_planes = 0
    for plane in PLANES:
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            features.append(torch.zeros(512))
            continue
        row = choices.iloc[0]
        paths = ordered_slices(image_root / study_uid / row.SeriesInstanceUID)
        # shortcut: three slices from one series per plane; expand coverage before competitive training.
        indices = np.unique(np.rint(np.linspace(0.25, 0.75, 3) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i]) for i in indices]).to(device)
        features.append(encoder(batch).mean(dim=0).cpu())
        used_planes += 1
    assert used_planes, f"No recognized anatomical planes for {study_uid}"
    result = torch.cat(features)
    assert result.shape == (1536,) and torch.isfinite(result).all()
    return result


X = torch.stack([study_features(uid, series, DATA / "train_series")
                 for uid in tqdm(gold.StudyInstanceUID, desc="MRI features: labeled studies")])
assert X.shape == (len(gold), 1536) and X.std() > 0
torch.save({"features": X, "study_ids": gold.StudyInstanceUID.tolist()}, OUT / "gold_features.pt")



## 4. Train a small classification head; the image encoder stays frozen.

In [ ]:
mean = X[train_idx].mean(dim=0)
std = X[train_idx].std(dim=0).clamp_min(0.01)
X = ((X - mean) / std).to(device)
targets = torch.from_numpy(y).to(device)
head = nn.Linear(1536, len(labels)).to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.1)
positives = targets[train_idx].sum(dim=0)
criterion = nn.BCEWithLogitsLoss(pos_weight=(len(train_idx) - positives) / positives)
best_loss = float("inf")
best_state = None
history = []
for epoch in range(1, EPOCHS + 1):
    head.train()
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(head(X[train_idx]), targets[train_idx])
    assert torch.isfinite(loss), "Non-finite training loss."
    loss.backward()
    optimizer.step()
    head.eval()
    with torch.inference_mode():
        logits = head(X[valid_idx])
        val_loss = nn.functional.binary_cross_entropy_with_logits(logits, targets[valid_idx]).item()
        probabilities = logits.sigmoid().cpu().numpy()
    auc = roc_auc_score(y[valid_idx], probabilities, average="macro")
    history.append({"epoch": epoch, "train_loss": loss.item(), "valid_loss": val_loss, "macro_auc": auc})
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        best_epoch = epoch
    if epoch == 1 or epoch % 20 == 0:
        print(history[-1])

assert best_state is not None
head.load_state_dict(best_state)
head.eval()
with torch.inference_mode():
    validation_predictions = head(X[valid_idx]).sigmoid().cpu().numpy()
assert np.isfinite(validation_predictions).all()
metrics = {label: float(roc_auc_score(y[valid_idx, i], validation_predictions[:, i]))
           for i, label in enumerate(labels)}
metrics["macro_auc"] = float(np.mean(list(metrics.values())))
metrics["best_epoch"] = best_epoch
metrics["train_studies"] = len(train_idx)
metrics["validation_studies"] = len(valid_idx)
metrics["note"] = "Pilot only: 15 validation studies; not a leaderboard estimate."
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
predictions = pd.DataFrame(validation_predictions, columns=labels)
predictions.insert(0, "StudyInstanceUID", gold.iloc[valid_idx].StudyInstanceUID.to_numpy())
predictions.to_csv(OUT / "validation_predictions.csv", index=False)
torch.save({"encoder": {k: v.cpu() for k, v in encoder.state_dict().items()},
            "head": best_state, "mean": mean, "std": std, "labels": labels,
            "planes": list(PLANES), "seed": SEED, "best_epoch": best_epoch}, OUT / "baseline.pt")
print(json.dumps(metrics, indent=2))
print("Saved baseline.pt, metrics.json, history.csv, split.csv and validation_predictions.csv")
print("Training pilot complete. An offline inference notebook is needed for competition submission.")
